In [1]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")


GPU Available: True
GPU Name: Tesla T4


In [2]:
import os
from pathlib import Path

print("Locating LLVIP dataset...")

# Base input path
input_base = Path('/kaggle/input/llvip-rgb-thermal-yolo-format')

# List all contents to find correct structure
print("\nExploring dataset structure:")
for item in input_base.rglob('*'):
    if item.is_dir():
        level = len(item.relative_to(input_base).parts)
        if level <= 3:  # Only show first 3 levels
            indent = "  " * level
            print(f"{indent}{item.name}/")

# Find the processed folder
if (input_base / 'processed').exists():
    data_path = input_base / 'processed'
    print(f"\n✓ Found 'processed' folder")
elif (input_base / 'rgb').exists():
    data_path = input_base
    print(f"\n✓ Dataset at root level")
else:
    # Search for rgb folder recursively
    rgb_folders = list(input_base.rglob('rgb'))
    if rgb_folders:
        data_path = rgb_folders[0].parent
        print(f"\n✓ Found dataset at: {data_path}")
    else:
        print("\n✗ Could not find dataset structure")
        data_path = input_base

print(f"\nUsing path: {data_path}")

# Check if rgb and thermal folders exist
rgb_exists = (data_path / 'rgb').exists()
thermal_exists = (data_path / 'thermal').exists()

print(f"\nFolder check:")
print(f"  RGB folder exists: {rgb_exists}")
print(f"  Thermal folder exists: {thermal_exists}")

if rgb_exists and thermal_exists:
    # Verify both modalities
    rgb_train = len(list((data_path / 'rgb' / 'images' / 'train').glob('*.jpg')))
    rgb_val = len(list((data_path / 'rgb' / 'images' / 'val').glob('*.jpg')))
    thermal_train = len(list((data_path / 'thermal' / 'images' / 'train').glob('*.jpg')))
    thermal_val = len(list((data_path / 'thermal' / 'images' / 'val').glob('*.jpg')))

    print("\nDataset Verification:")
    print("="*50)
    print(f"RGB Training images: {rgb_train:,}")
    print(f"RGB Validation images: {rgb_val:,}")
    print(f"Thermal Training images: {thermal_train:,}")
    print(f"Thermal Validation images: {thermal_val:,}")
    print("="*50)

    if rgb_train == 12025 and thermal_train == 12025:
        print("\n✓ Dataset ready for training!")
    elif rgb_train > 0 and thermal_train > 0:
        print(f"\n✓ Dataset found but counts differ from expected")
        print(f"   This is OK - proceeding with {rgb_train} training images")
    else:
        print("\n✗ Warning: No images found in expected locations")
        
        # Debug: show what's actually in the folders
        print("\nDebug - checking actual folder contents:")
        if (data_path / 'rgb').exists():
            print(f"  Contents of {data_path / 'rgb'}:")
            for item in (data_path / 'rgb').iterdir():
                print(f"    - {item.name}")
else:
    print("\n✗ RGB or Thermal folders not found")
    print("\nPlease check:")
    print("  1. Dataset was uploaded correctly")
    print("  2. ZIP file contained 'processed/rgb/' and 'processed/thermal/' folders")


Locating LLVIP dataset...

Exploring dataset structure:
  processed/
    rgb/
    thermal/
      labels/
      images/
      labels/
      images/

✓ Found 'processed' folder

Using path: /kaggle/input/llvip-rgb-thermal-yolo-format/processed

Folder check:
  RGB folder exists: True
  Thermal folder exists: True

Dataset Verification:
RGB Training images: 12,025
RGB Validation images: 3,463
Thermal Training images: 12,025
Thermal Validation images: 3,463

✓ Dataset ready for training!


In [4]:
# Install required packages
print("Installing Ultralytics...")
!pip install -q ultralytics

print("\nVerifying installation...")
import torch
from ultralytics import YOLO

print("="*70)
print("INSTALLATION COMPLETE")
print("="*70)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("="*70)


Installing Ultralytics...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.5 MB/s eta 0:00:00a 0:00:01

Verifying installation...
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
INSTALLATION COMPLETE
PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.83 GB


In [5]:
from ultralytics import YOLO
import torch
from pathlib import Path

# GPU Check
print("="*70)
print("SYSTEM CHECK")
print("="*70)
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"CUDA Version: {torch.version.cuda}")
print("="*70)

# Dataset path (from previous cell)
data_path = Path('/kaggle/input/llvip-rgb-thermal-yolo-format/processed')

# Create YAML configuration files
rgb_yaml = Path('/kaggle/working/rgb_dataset.yaml')
thermal_yaml = Path('/kaggle/working/thermal_dataset.yaml')

# RGB configuration
rgb_config = f"""path: {data_path / 'rgb'}
train: images/train
val: images/val

names:
  0: person

nc: 1
"""

# Thermal configuration
thermal_config = f"""path: {data_path / 'thermal'}
train: images/train
val: images/val

names:
  0: person

nc: 1
"""

# Write configurations
rgb_yaml.write_text(rgb_config)
thermal_yaml.write_text(thermal_config)

print(f"\n✓ Configuration files created")
print(f"  RGB: {rgb_yaml}")
print(f"  Thermal: {thermal_yaml}")

# ============================================
# PART 1: TRAIN RGB BASELINE
# ============================================

print("\n" + "="*70)
print("TRAINING RGB BASELINE")
print("="*70)
print("Model: YOLOv8n")
print("Training images: 12,025")
print("Validation images: 3,463")
print("Epochs: 100")
print("Estimated time: 2-3 hours")
print("="*70)

model_rgb = YOLO('yolov8n.pt')

results_rgb = model_rgb.train(
    data=str(rgb_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    
    project='/kaggle/working/models',
    name='rgb_baseline',
    
    optimizer='SGD',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    
    val=True,
    save=True,
    save_period=10,
    plots=True,
    patience=20,
    workers=8,
    verbose=True
)

print("\n" + "="*70)
print("RGB BASELINE COMPLETE")
print("="*70)

# Display RGB results
rgb_metrics = results_rgb.results_dict
print(f"\nRGB Results:")
print(f"  mAP@0.5: {rgb_metrics.get('metrics/mAP50(B)', 0):.4f}")
print(f"  mAP@0.5:0.95: {rgb_metrics.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"  Precision: {rgb_metrics.get('metrics/precision(B)', 0):.4f}")
print(f"  Recall: {rgb_metrics.get('metrics/recall(B)', 0):.4f}")

# ============================================
# PART 2: TRAIN THERMAL BASELINE
# ============================================

print("\n" + "="*70)
print("TRAINING THERMAL BASELINE")
print("="*70)
print("Model: YOLOv8n")
print("Training images: 12,025")
print("Validation images: 3,463")
print("Epochs: 100")
print("Estimated time: 2-3 hours")
print("="*70)

model_thermal = YOLO('yolov8n.pt')

results_thermal = model_thermal.train(
    data=str(thermal_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    
    project='/kaggle/working/models',
    name='thermal_baseline',
    
    optimizer='SGD',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    
    val=True,
    save=True,
    save_period=10,
    plots=True,
    patience=20,
    workers=8,
    verbose=True
)

print("\n" + "="*70)
print("THERMAL BASELINE COMPLETE")
print("="*70)

# Display thermal results
thermal_metrics = results_thermal.results_dict
print(f"\nThermal Results:")
print(f"  mAP@0.5: {thermal_metrics.get('metrics/mAP50(B)', 0):.4f}")
print(f"  mAP@0.5:0.95: {thermal_metrics.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"  Precision: {thermal_metrics.get('metrics/precision(B)', 0):.4f}")
print(f"  Recall: {thermal_metrics.get('metrics/recall(B)', 0):.4f}")

# ============================================
# PART 3: COMPARISON
# ============================================

print("\n" + "="*70)
print("BASELINE COMPARISON")
print("="*70)

rgb_map = rgb_metrics.get('metrics/mAP50(B)', 0)
thermal_map = thermal_metrics.get('metrics/mAP50(B)', 0)

print(f"\nPerformance Summary:")
print(f"  RGB mAP@0.5:     {rgb_map:.4f}")
print(f"  Thermal mAP@0.5: {thermal_map:.4f}")
print(f"  Difference:      {abs(rgb_map - thermal_map):.4f}")

if thermal_map > rgb_map:
    improvement = ((thermal_map - rgb_map) / rgb_map) * 100
    print(f"\n→ Thermal outperforms RGB by {improvement:.2f}%")
else:
    improvement = ((rgb_map - thermal_map) / thermal_map) * 100
    print(f"\n→ RGB outperforms Thermal by {improvement:.2f}%")

print("\n" + "="*70)
print("ALL TRAINING COMPLETE")
print("="*70)
print("\nModel locations:")
print("  RGB: /kaggle/working/models/rgb_baseline/weights/best.pt")
print("  Thermal: /kaggle/working/models/thermal_baseline/weights/best.pt")
print("\nResults saved to:")
print("  /kaggle/working/models/rgb_baseline/")
print("  /kaggle/working/models/thermal_baseline/")


SYSTEM CHECK
GPU Available: True
GPU Name: Tesla T4
CUDA Version: 12.6

✓ Configuration files created
  RGB: /kaggle/working/rgb_dataset.yaml
  Thermal: /kaggle/working/thermal_dataset.yaml

TRAINING RGB BASELINE
Model: YOLOv8n
Training images: 12,025
Validation images: 3,463
Epochs: 100
Estimated time: 2-3 hours
Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rgb_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640,